# 🚀 IndexTTS-2.5 · 3 步部署 (Colab)

运行前: `运行时 → 更改运行时类型 → 硬件加速器 = GPU` (T4 即可)。

**步骤 1** 一键安装 → **步骤 2** 后台启动服务(API+WebUI, 自动打印公网 URL) → **步骤 3** 查看实时日志。

> 仓库地址: `https://github.com/infinite-gaming-studio/index-tts`（本仓库）
>
> 下载慢可先运行 `%env HF_ENDPOINT=https://hf-mirror.com`；也可改用 ModelScope: 在步骤1前运行 `%env MODEL_SOURCE=modelscope`
>
> 步骤 2 默认启动 **API + WebUI 双服务**（`SERVICE=both`）：API 在 `{公网URL}/api/tts`（文档 `/docs`），WebUI 在 `{公网URL}/`

In [ ]:
# ===== 步骤 1/3: 一键部署（克隆本仓库 → 装依赖 → 下载 IndexTTS-2.5 权重）=====
REPO_URL = "https://github.com/infinite-gaming-studio/index-tts.git"

!git clone --depth 1 $REPO_URL index-tts
%cd index-tts
!bash deploy/scripts/setup.sh

In [ ]:
# ===== 步骤 2/3: 后台启动服务 (API + WebUI) + ngrok 公网隧道 =====
# 默认 SERVICE=both (API 端口 8000 + WebUI 挂载 /ui), 可用 %env SERVICE=api|webui 切换
# 默认 TUNNEL=ngrok (需 %env NGROK_TOKEN=你的authtoken); 想用 cloudflare 免注册: %env TUNNEL=cf
# 服务日志: api.log (API+WebUI), 心跳日志: keepalive.log
import os, re, time

os.environ["SERVICE"] = "both"  # api | webui | both
os.system("nohup bash deploy/scripts/serve.sh > serve_console.log 2>&1 &")

# 轮询等待公网 URL (最多约 10 分钟), 期间可切到步骤 3 看日志
# 兼容 ngrok (.ngrok-free.app / ngrok.app / ngrok.io) 与 cloudflare (.trycloudflare.com)
url_pattern = re.compile(r"https://[a-z0-9-]+\.(?:ngrok-free\.app|ngrok\.app|ngrok\.io|trycloudflare\.com)")
found = None
for i in range(120):
    time.sleep(5)
    try:
        log = open("serve_console.log", encoding="utf-8", errors="ignore").read()
    except FileNotFoundError:
        continue
    m = url_pattern.search(log)
    if m:
        found = m.group(0)
        break

if found:
    print(f"✅ 公网地址: {found}")
    print(f"   API 合成:  POST {found}/api/tts   (multipart: text + spk_audio)")
    print(f"   健康检查:  GET  {found}/api/health")
    print(f"   API 文档:  {found}/docs")
    print(f"   WebUI:     {found}/")
else:
    print("⚠️ 尚未获取到公网 URL, 请运行步骤 3 查看日志排查:")
    os.system("tail -40 serve_console.log")

In [ ]:
# ===== 步骤 3/3: 查看实时日志 (排查问题) =====
# 中断本单元格仅停止查看日志, 不影响后台服务。
# 默认 SERVICE=both, 日志在 api.log (WebUI 模式才用 webui.log)
!bash deploy/scripts/logs.sh api -f

In [ ]:
# ===== 防 Colab 空闲断连: 定时点击"连接"按钮 (需保持本标签页打开) =====
# Colab 免费版约 90 分钟无操作会断连; 此脚本每 60s 自动点一次连接按钮保活。
# 停止: 中断本单元格; 重新运行即可恢复。
from IPython.display import display, Javascript

display(Javascript("""
function clickConnect(){
  const btn = document.querySelector('colab-connect-button');
  if (btn) btn.click();
}
setInterval(clickConnect, 60000);
"""))
print("✅ 防断连脚本已启动 (每 60s 自动重连; 请保持浏览器标签页打开)")

## (可选) 合成验证

以下为**默认注释**的参考代码，如需使用请取消注释后运行。

In [ ]:
# ===== (可选) 初始化 + 语音克隆合成 =====
# from indextts.utils.examples_downloader import ensure_examples_available
# ensure_examples_available()
#
# from indextts.infer_v2_5 import IndexTTS2
# tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)
#
# tts.infer(
#     spk_audio_prompt="examples/voice_01.wav",
#     text="你好，我是 IndexTTS-2.5，欢迎测试多语言语音合成。",
#     lang="ZH",
#     output_path="output_zh.wav",
#     verbose=True,
# )

In [ ]:
# ===== (可选) 播放 + 下载结果 =====
# from IPython.display import Audio
# from google.colab import files
#
# Audio("output_zh.wav")
# files.download("output_zh.wav")